# Step 12 — Rules vs LLM on FINAL's validation units

**Input (read only):** `data/output/11_rules_on_final.parquet` (notebook 11); from the FINAL pipeline `data/output/06_buildings_classified.gpkg` (the LLM's answers), `data/experimental_extract/09_validation.gpkg` (layer `units`: the truth per building, written by FINAL's notebook 09), and FINAL's `config.py` and `lib/validation.py`
**Output:** none — the tables below

Both classifiers are scored on the same units, against the same truth, by the same code as FINAL's notebook 09:

- **Units:** the validation units on buildings FINAL's step 04 kept (1,101), keyed by `building_id`. Bosserhof is scored on the 1,025 with a class in the truth, activities on the 1,034 with an activity truth.
- **Bosserhof:** *exact* (the answer is an accepted class) and *by headline* (the same main group), after FINAL's `norm_class`. A building the rules leave without a class counts as a miss.
- **Activities:** micro precision / recall / F1 over the six categories besides Workers. Headline score: the labels collapsed by the **full old merge** (`VALIDATION_OLD_LABEL_MERGE`: errands → Retail_Daily; sports, meetup, lessons → Leisure). Second line: FINAL's `MID_LABEL_TO_ACTIVITY`, the map of FINAL's notebook 09.
- **Arms:** the LLM (FINAL 06), the rules with sites (the main run), the rules with own POIs only.

**Read the scores with this in mind.** The truth is a human check of the ORIGINAL pipeline's LLM answers, and the old run requested the same model as FINAL (gpt-oss-120b). Where the validator agreed, the truth *is* that old answer: on these units (*old answer right*) a model like the old one has an edge. Where the validator rejected it (*old answer wrong*), the units are exactly the old model's mistakes, which tilts the other way. Neither split is neutral; both are shown, and only a blind sample could settle it. Differences are tested pairwise, because both classifiers answer the same units. The LLM answered each building once, so its run-to-run spread is not in these numbers.


In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import hashlib
import importlib.util
import math
import subprocess

import numpy as np
import pandas as pd
import pyogrio


def _load(name, path):
    """a module of the FINAL pipeline, by file path and under its own name, so it never shadows this config"""
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


FC = _load('final_config', FINAL_CONFIG_FILE)          # FINAL's catalogue and label maps
V = _load('final_validation', FINAL_VALIDATION_LIB)     # FINAL's scoring functions, as notebook 09 uses them

SIX = list(FC.VALIDATION_SCORED_CATEGORIES)
S6 = frozenset(SIX)
HEAD = V.class_headlines(FC.LLM_BOSSERHOF_SUBCATEGORIES, FC.BOSSERHOF_NORMALIZATION_MAP)
MAPS = {'old merge': FC.VALIDATION_OLD_LABEL_MERGE, 'FINAL map': FC.MID_LABEL_TO_ACTIVITY}
ARMS = {'LLM (FINAL 06)': '06', 'rules, sites (main)': 'sites', 'rules, own POIs': 'own'}
PAIRS = (('LLM (FINAL 06)', 'rules, sites (main)'), ('rules, sites (main)', 'rules, own POIs'))


def _sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()[:12]


def _git(*args):
    return subprocess.run(['git', '-C', str(FINAL_DIR), *args], capture_output=True, text=True).stdout.strip()


print('What is compared:')
for _f in (RULES_ON_FINAL_FILE, ROOT / 'rule_utils.py', FINAL_CLASSIFIED_FILE, FINAL_VALIDATION_FILE,
           FINAL_CONFIG_FILE, FINAL_VALIDATION_LIB):
    print(f'  {_sha(_f)}  {_f}')
_dirty = _git('status', '--porcelain', '--', 'config.py', 'lib/validation.py').splitlines()
print(f"FINAL at {_git('rev-parse', '--short', 'HEAD')} on {_git('branch', '--show-current')}; "
      f"not committed there: {'; '.join(_dirty) if _dirty else 'nothing'}")

What is compared:
  7ad3d133dc8a  ..\Capacity_Calculation-pipeline-RULES\data\output\11_rules_on_final.parquet
  80b9f326fc91  ..\Capacity_Calculation-pipeline-RULES\rule_utils.py
  43e55cd7c5d5  ..\Capacity_Calculation-pipeline-FINAL\data\output\06_buildings_classified.gpkg
  0f6ab22f3352  ..\Capacity_Calculation-pipeline-FINAL\data\experimental_extract\09_validation.gpkg
  e3f282ad4110  ..\Capacity_Calculation-pipeline-FINAL\config.py
  72ae4ceb9cfc  ..\Capacity_Calculation-pipeline-FINAL\lib\validation.py


FINAL at ecca7d5 on final-pipeline; not committed there: M config.py; ?? lib/validation.py


## 1. The units, the truth and the three answers

In [2]:
def _set(text):
    """a set stored as 'a;b' text -> frozenset; a missing cell stays None (no truth); '' is the empty set"""
    return frozenset(s for s in text.split(';') if s) if isinstance(text, str) else None


def _cls(c):
    """FINAL's norm_class; no class stays None (scored as a miss), never the string 'none'"""
    return V.norm_class(c, FC.BOSSERHOF_NORMALIZATION_MAP) if isinstance(c, str) and c else None


def _obj(values, index):
    """object column that keeps None and frozensets as they are (pandas 3 would turn None in text into NaN)"""
    return pd.Series(list(values), index=index, dtype=object)


# FINAL is only read: its GeoPackages are opened immutable, so SQLite writes no side files next to them (06 is in WAL mode)
units = pyogrio.read_dataframe(FINAL_VALIDATION_FILE, layer='units', read_geometry=False, IMMUTABLE='YES')
llm = pyogrio.read_dataframe(FINAL_CLASSIFIED_FILE, layer='buildings', read_geometry=False, IMMUTABLE='YES',
                             columns=['building_id', 'llm_labels', 'bosserhof_class', 'evidence']).set_index('building_id')
rules = pd.read_parquet(RULES_ON_FINAL_FILE).set_index('building_id')
assert rules.index.is_unique and set(rules.index) == set(llm.index), 'the rules and the LLM did not answer the same buildings'

ev = units[units['kept']].copy()
assert ev['building_id'].isin(llm.index).all(), 'a kept unit is missing from 06'
for _c in ('act_truth', 'cls_truth', 'old_set'):
    ev[_c] = _obj(ev[_c].map(_set), ev.index)
ev['old_class'] = _obj((c if isinstance(c, str) and c else None for c in ev['old_class']), ev.index)
ev['cls_06'] = _obj((_cls(c) for c in ev['building_id'].map(llm['bosserhof_class'])), ev.index)
ev['lab_06'] = ev['building_id'].map(llm['llm_labels'])
ev['evidence'] = ev['building_id'].map(llm['evidence'])
for _r in ('sites', 'own'):
    ev[f'cls_{_r}'] = _obj((_cls(c) for c in ev['building_id'].map(rules[f'cls_{_r}'])), ev.index)
    ev[f'lab_{_r}'] = ev['building_id'].map(rules[f'labels_{_r}'])
    ev[f'layer_{_r}'] = ev['building_id'].map(rules[f'layer_{_r}'])

# the 06 answers stored in 09_validation.gpkg must be the ones read here: 09 is in step with 06
_stored = ev['building_id'].map(units.set_index('building_id')['cls_06'])
assert all((s if isinstance(s, str) else None) == c for s, c in zip(_stored, ev['cls_06'])), \
    '09_validation.gpkg is out of step with 06 - re-run FINAL notebook 09 first'
_off = sorted({c for a in ARMS.values() for c in ev[f'cls_{a}'] if c is not None} - set(HEAD))
assert not _off, f'answers outside the catalogue after normalisation: {_off}'

bh = ev[[isinstance(t, frozenset) and len(t) > 0 for t in ev['cls_truth']]].copy()
bh['old_right'] = [o in t for o, t in zip(bh['old_class'], bh['cls_truth'])]
ac = ev[[isinstance(t, frozenset) for t in ev['act_truth']]].copy()
ac['old_right'] = [(t & S6) == ((o or frozenset()) & S6) for t, o in zip(ac['act_truth'], ac['old_set'])]
print(f'{len(ev):,} units on kept buildings. Bosserhof scored on {len(bh):,} (old answer right on '
      f"{int(bh['old_right'].sum())}), activities on {len(ac):,} (old answer right on {int(ac['old_right'].sum())})")

_s = V.class_scores(bh, 'cls_truth', 'cls_06', HEAD)
_d, _ = V.set_scores(ac['act_truth'], [V.to_categories(s, FC.MID_LABEL_TO_ACTIVITY) for s in ac['lab_06']], SIX)
print(f"check against FINAL's notebook 09 (64.8 / 78.5 and F1 71.2 there): 06 Bosserhof {_s['exact_%']:.1f} % exact, "
      f"{_s['headline_%']:.1f} % by headline; activity F1 {100 * _d.set_index('category').loc['micro', 'f1']:.1f}")

1,101 units on kept buildings. Bosserhof scored on 1,025 (old answer right on 845), activities on 1,034 (old answer right on 703)
check against FINAL's notebook 09 (64.8 / 78.5 and F1 71.2 there): 06 Bosserhof 64.8 % exact, 78.5 % by headline; activity F1 71.2


## 2. Bosserhof class

*Exact*: the answer is one of the accepted classes. *By headline*: it lies in the same main group. A *coarser* miss answers the main group where the truth names a subcategory; a *finer* one answers a subcategory where the truth accepts only the main group.

In [3]:
def _subsets(frame, strict_col):
    """the rows of the tables: all units, the strict set, and whether the truth is the old LLM answer"""
    return {'all units': frame,
            'strict (as first decided)': frame[frame[strict_col]],
            'old answer right (truth = old LLM answer)': frame[frame['old_right']],
            'old answer wrong (truth typed by validator)': frame[~frame['old_right']]}


_rows = []
for _name, _f in _subsets(bh, 'strict_cls').items():
    _r = {('', 'units'): len(_f)}
    for arm, a in ARMS.items():
        _sc = V.class_scores(_f, 'cls_truth', f'cls_{a}', HEAD)
        _r[(arm, 'exact %')] = _sc['exact_%']
        _r[(arm, 'headline %')] = _sc['headline_%']
    _rows.append(pd.Series(_r, name=_name))
_tab = pd.DataFrame(_rows)
_tab[('', 'units')] = _tab[('', 'units')].astype(int)
print('Bosserhof accuracy (%):')
print(_tab.round(1).to_string())
print()
for arm, a in ARMS.items():
    _hits = [V.class_hits(t, c, HEAD) for t, c in zip(bh['cls_truth'], bh[f'cls_{a}'])]
    _none = [c is None for c in bh[f'cls_{a}']]
    _miss = [(h, n) for h, n in zip(_hits, _none) if not h[0]]
    _heads = sum(c is not None and HEAD.get(c) == c for c in bh[f'cls_{a}'])
    print(f'  ..  {arm}: {len(_miss)} exact misses - {sum(not h[1] and not n for h, n in _miss)} another headline, '
          f'{sum(h[1] and not h[2] and not h[3] for h, n in _miss)} a sibling, {sum(h[2] for h, n in _miss)} coarser, '
          f'{sum(h[3] for h, n in _miss)} finer, {sum(_none)} no class. It answers a bare headline on {_heads} units.')
_wr = bh[~bh['old_right']]
print()
for arm, a in ARMS.items():
    print(f'  ..  where the old answer was wrong ({len(_wr)} units), {arm} gives the rejected old class again on '
          f"{sum(c == o for c, o in zip(_wr[f'cls_{a}'], _wr['old_class']))}")

Bosserhof accuracy (%):
                                                  LLM (FINAL 06)            rules, sites (main)            rules, own POIs           
                                            units        exact % headline %             exact % headline %         exact % headline %
all units                                    1025           64.8       78.5                59.6       76.6            58.1       75.5
strict (as first decided)                     889           66.1       79.0                61.3       77.5            59.7       76.5
old answer right (truth = old LLM answer)     845           73.0       83.2                65.8       80.0            64.9       79.2
old answer wrong (truth typed by validator)   180           26.1       56.7                30.6       60.6            26.7       58.3

  ..  LLM (FINAL 06): 361 exact misses - 220 another headline, 55 a sibling, 56 coarser, 30 finer, 0 no class. It answers a bare headline on 203 units.
  ..  rules, sites 

In [4]:
def mcnemar(x, y):
    """exact two-sided McNemar test for paired hits: (right only by x, right only by y, p)"""
    x, y = np.asarray(x, bool), np.asarray(y, bool)
    b, c = int((x & ~y).sum()), int((~x & y).sum())
    n, k = b + c, min(b, c)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    return b, c, p


_rows = []
for x, y in PAIRS:
    for _name, _f in _subsets(bh, 'strict_cls').items():
        for _k, _what in ((0, 'exact'), (1, 'headline')):
            _hx = [V.class_hits(t, c, HEAD)[_k] for t, c in zip(_f['cls_truth'], _f[f'cls_{ARMS[x]}'])]
            _hy = [V.class_hits(t, c, HEAD)[_k] for t, c in zip(_f['cls_truth'], _f[f'cls_{ARMS[y]}'])]
            b, c, p = mcnemar(_hx, _hy)
            _rows.append({'x vs y': f'{x} vs {y}', 'units': _name, 'measure': _what,
                          'right only by x': b, 'right only by y': c, 'p': p})
print('paired: units one arm gets right and the other does not (exact two-sided McNemar test)')
print(pd.DataFrame(_rows).to_string(index=False, formatters={'p': lambda v: f'{v:.2g}'}))

paired: units one arm gets right and the other does not (exact two-sided McNemar test)
                                x vs y                                       units  measure  right only by x  right only by y       p
 LLM (FINAL 06) vs rules, sites (main)                                   all units    exact              159              106  0.0014
 LLM (FINAL 06) vs rules, sites (main)                                   all units headline              116               96    0.19
 LLM (FINAL 06) vs rules, sites (main)                   strict (as first decided)    exact              136               93  0.0054
 LLM (FINAL 06) vs rules, sites (main)                   strict (as first decided) headline               99               86    0.38
 LLM (FINAL 06) vs rules, sites (main)   old answer right (truth = old LLM answer)    exact              141               80 4.9e-05
 LLM (FINAL 06) vs rules, sites (main)   old answer right (truth = old LLM answer) headline               91 

## 3. Activities

The six categories besides Workers, micro-averaged over the units. Workers is left out as in FINAL's notebook 09: the rules add work to every building they give a class, and FINAL's work rule does the same for the LLM.

In [5]:
for a in ARMS.values():
    for m, mp in MAPS.items():
        ac[f'p_{a}_{m}'] = _obj((V.to_categories(s, mp) for s in ac[f'lab_{a}']), ac.index)


def _micro(frame, m):
    """micro P / R / F1 and the exact-set share per arm, in %"""
    out = {}
    for arm, a in ARMS.items():
        df, ex = V.set_scores(frame['act_truth'], frame[f'p_{a}_{m}'], SIX)
        mi = df.set_index('category').loc['micro']
        out[arm] = {'P': 100 * mi['precision'], 'R': 100 * mi['recall'], 'F1': 100 * mi['f1'], 'exact': 100 * ex}
    return pd.DataFrame(out).T


for m in MAPS:
    print(f'all {len(ac):,} units, labels through the {m} (%):')
    print(_micro(ac, m).round(1).to_string())
    print()
for m in MAPS:
    _sub = _subsets(ac, 'strict_act')
    _f1 = pd.DataFrame({k: _micro(f, m)['F1'] for k, f in _sub.items()}).round(1).astype(object)
    _f1.loc['units'] = [len(f) for f in _sub.values()]
    print(f'micro F1 by subset, {m} (%):')
    print(_f1.to_string())
    print()
for arm in list(ARMS)[:2]:
    _df, _ex = V.set_scores(ac['act_truth'], ac[f'p_{ARMS[arm]}_old merge'], SIX)
    print(f'per category, {arm}, old merge (exact six-category set {100 * _ex:.1f} %):')
    print(_df.round(3).to_string(index=False))
    print()

all 1,034 units, labels through the old merge (%):
                        P     R    F1  exact
LLM (FINAL 06)       60.4  75.4  67.1   58.4
rules, sites (main)  62.4  68.8  65.4   58.8
rules, own POIs      61.7  66.6  64.1   57.7

all 1,034 units, labels through the FINAL map (%):


                        P     R    F1  exact
LLM (FINAL 06)       79.1  64.8  71.2   66.4
rules, sites (main)  83.6  64.1  72.6   68.4
rules, own POIs      83.5  61.9  71.1   67.2



micro F1 by subset, old merge (%):
                    all units strict (as first decided) old answer right (truth = old LLM answer) old answer wrong (truth typed by validator)
LLM (FINAL 06)           67.1                      67.5                                      83.9                                        38.1
rules, sites (main)      65.4                      66.5                                      81.6                                        35.9
rules, own POIs          64.1                      65.3                                      80.3                                        34.4
units                    1034                       890                                       703                                         331



micro F1 by subset, FINAL map (%):
                    all units strict (as first decided) old answer right (truth = old LLM answer) old answer wrong (truth typed by validator)
LLM (FINAL 06)           71.2                      71.4                                      83.3                                        45.1
rules, sites (main)      72.6                      73.5                                      84.0                                        45.8
rules, own POIs          71.1                      72.2                                      83.1                                        43.0
units                    1034                       890                                       703                                         331

per category, LLM (FINAL 06), old merge (exact six-category set 58.4 %):
        category  support  predicted  tp  fp  fn  precision  recall    f1
          School       52         47  45   2   7      0.957   0.865 0.909
      University       27        

In [6]:
def _counts(frame, col):
    """per unit: true positives, false positives, false negatives over the six categories"""
    return np.array([[len(t & p & S6), len((p & S6) - t), len((t & S6) - p)]
                     for t, p in zip(frame['act_truth'], frame[col])])


def _f1(c):
    return 2 * c[..., 0] / (2 * c[..., 0] + c[..., 1] + c[..., 2])


def boot_diff(frame, col_x, col_y):
    """micro F1 of x minus y in points, a 95 % percentile interval and a two-sided p from a paired bootstrap"""
    cx, cy = _counts(frame, col_x), _counts(frame, col_y)
    rng = np.random.default_rng(RULES_VS_LLM_SEED)
    idx = rng.integers(0, len(frame), size=(RULES_VS_LLM_BOOTSTRAP, len(frame)))
    d = 100 * (_f1(cx[idx].sum(axis=1)) - _f1(cy[idx].sum(axis=1)))
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return 100 * (_f1(cx.sum(axis=0)) - _f1(cy.sum(axis=0))), lo, hi, p


_rows = []
for m in MAPS:
    for x, y in PAIRS:
        for _name, _f in _subsets(ac, 'strict_act').items():
            d, lo, hi, p = boot_diff(_f, f'p_{ARMS[x]}_{m}', f'p_{ARMS[y]}_{m}')
            _rows.append({'labels through': m, 'x - y': f'{x} - {y}', 'units': _name, 'n': len(_f),
                          'F1 difference': round(d, 1), '95 % interval': f'{lo:+.1f} .. {hi:+.1f}', 'p': p})
print(f'paired: micro F1 of x minus y, in points ({RULES_VS_LLM_BOOTSTRAP:,} bootstrap resamples of the units)')
print(pd.DataFrame(_rows).to_string(index=False, formatters={
    'p': lambda v: f'< {2 / RULES_VS_LLM_BOOTSTRAP:.3f}' if v == 0 else f'{v:.2g}'}))     # p moves in steps of 2 / B

paired: micro F1 of x minus y, in points (2,000 bootstrap resamples of the units)
labels through                                 x - y                                       units    n  F1 difference 95 % interval       p
     old merge  LLM (FINAL 06) - rules, sites (main)                                   all units 1034            1.6  -0.4 .. +3.7    0.12
     old merge  LLM (FINAL 06) - rules, sites (main)                   strict (as first decided)  890            1.0  -1.1 .. +3.1    0.38
     old merge  LLM (FINAL 06) - rules, sites (main)   old answer right (truth = old LLM answer)  703            2.3  -0.3 .. +4.9   0.082
     old merge  LLM (FINAL 06) - rules, sites (main) old answer wrong (truth typed by validator)  331            2.2  -1.1 .. +5.2     0.2
     old merge rules, sites (main) - rules, own POIs                                   all units 1034            1.4  +0.6 .. +2.2   0.002
     old merge rules, sites (main) - rules, own POIs                   strict (as fi

## 4. Where each rule layer decides — on the scored units

The rules answer from POI tags, from the ALKIS code, or from the OSM footprint. The validation sample holds far more POI buildings than the region, so the same split is shown for every kept building (last column).

In [7]:
def _cls_table(frame, by):
    _rows = {}
    for k, f in frame.groupby(by):
        r = {'units': len(f)}
        for arm in list(ARMS)[:2]:
            sc = V.class_scores(f, 'cls_truth', f'cls_{ARMS[arm]}', HEAD)
            r[f'{arm}: exact %'] = sc['exact_%']
            r[f'{arm}: headline %'] = sc['headline_%']
        _rows[k] = r
    t = pd.DataFrame(_rows).T
    t['units'] = t['units'].astype(int)
    return t


_lay = _cls_table(bh, 'layer_sites')
_lay['share of kept buildings %'] = 100 * rules['layer_sites'].value_counts().reindex(_lay.index).fillna(0) / len(rules)
print('Bosserhof by the layer that decided the rules (sites run):')
print(_lay.round(1).to_string())
print()
_ev = _cls_table(bh, 'evidence')
_ev['share of kept buildings %'] = 100 * llm['evidence'].value_counts().reindex(_ev.index).fillna(0) / len(llm)
print("Bosserhof by the LLM record's evidence group:")
print(_ev.round(1).to_string())
print()
_rows = {}
for k, f in ac.groupby('layer_sites'):
    _rows[k] = {'units': len(f), **{f'{arm}: F1 %': _micro(f, 'old merge').loc[arm, 'F1'] for arm in list(ARMS)[:2]}}
_t = pd.DataFrame(_rows).T
_t['units'] = _t['units'].astype(int)
print('activities (old merge) by the layer that decided the rules:')
print(_t.round(1).to_string())

Bosserhof by the layer that decided the rules (sites run):
              units  LLM (FINAL 06): exact %  LLM (FINAL 06): headline %  rules, sites (main): exact %  rules, sites (main): headline %  share of kept buildings %
alkis_code      233                     41.2                        62.2                          38.2                             63.5                       54.5
osm_fallback      5                     60.0                        80.0                          20.0                             40.0                        2.2
poi_tags        787                     71.8                        83.4                          66.2                             80.7                       42.7

Bosserhof by the LLM record's evidence group:
             units  LLM (FINAL 06): exact %  LLM (FINAL 06): headline %  rules, sites (main): exact %  rules, sites (main): headline %  share of kept buildings %
class_only      45                     24.4                        44.4         

activities (old merge) by the layer that decided the rules:
              units  LLM (FINAL 06): F1 %  rules, sites (main): F1 %
alkis_code      238                  38.3                       29.3
osm_fallback      5                  50.0                        0.0
poi_tags        791                  70.8                       68.3


## 5. The whole region — how often the two agree

Every building FINAL's step 04 kept, rules (sites run) against the LLM. No truth is involved: agreement is not accuracy, but it shows where the two classifiers part ways outside the sample.

In [8]:
reg = llm.join(rules, how='inner')
_c06 = [_cls(c) for c in reg['bosserhof_class']]
_cs = [_cls(c) for c in reg['cls_sites']]
reg['same'] = [a == b for a, b in zip(_c06, _cs)]
reg['same_head'] = [b is not None and HEAD.get(a) == HEAD.get(b) for a, b in zip(_c06, _cs)]
reg['rules_none'] = [b is None for b in _cs]


def _agree(f):
    return pd.Series({'buildings': len(f), 'same class %': 100 * f['same'].mean(),
                      'same headline %': 100 * f['same_head'].mean(), 'rules: no class %': 100 * f['rules_none'].mean()})


_t = pd.DataFrame({'every kept building': _agree(reg),
                   **{f'rules layer: {k}': _agree(f) for k, f in reg.groupby('layer_sites')},
                   **{f'LLM evidence: {k}': _agree(f) for k, f in reg.groupby('evidence')}}).T
_t['buildings'] = _t['buildings'].astype(int)
print('Bosserhof, rules (sites) vs LLM:')
print(_t.round(1).to_string())
print()
_p06 = [V.to_categories(s, MAPS['old merge']) for s in reg['llm_labels']]
_ps = [V.to_categories(s, MAPS['old merge']) for s in reg['labels_sites']]
_rows = {}
for c in SIX:
    x = np.array([c in p for p in _p06])
    y = np.array([c in p for p in _ps])
    _rows[c] = {'both': int((x & y).sum()), 'only LLM': int((x & ~y).sum()), 'only rules': int((~x & y).sum()),
                'overlap (both / either) %': 100 * (x & y).sum() / max((x | y).sum(), 1)}
_t = pd.DataFrame(_rows).T
_t[['both', 'only LLM', 'only rules']] = _t[['both', 'only LLM', 'only rules']].astype(int)
print('activities (old merge), buildings carrying each category:')
print(_t.round(1).to_string())
print(f"  ..  the same six-category set on {100 * np.mean([(a & S6) == (b & S6) for a, b in zip(_p06, _ps)]):.1f} % "
      'of the kept buildings')
print()
_d = pd.Series([f'{b or "(no class)"}  ->  {a}' for a, b, s in zip(_c06, _cs, reg['same']) if not s])
print(f'the most frequent disagreements (rules class -> LLM class), of {len(_d):,}:')
print(_d.value_counts().head(15).to_string())

Bosserhof, rules (sites) vs LLM:
                           buildings  same class %  same headline %  rules: no class %
every kept building            39786          50.0             67.0                5.2
rules layer: alkis_code        21683          37.3             58.5                8.4
rules layer: no_signal           233           0.0              0.0              100.0
rules layer: osm_fallback        876          43.3             55.9                0.0
rules layer: poi_tags          16994          67.3             79.3                0.0
LLM evidence: class_only        6052          43.4             71.6                0.0
LLM evidence: name_only         1467          30.5             51.2                0.0
LLM evidence: poi_or_site      30103          53.5             67.1                6.8
LLM evidence: tag_only          2164          33.3             62.8                0.1



activities (old merge), buildings carrying each category:
                  both  only LLM  only rules  overlap (both / either) %
School            1333        52         129                       88.0
University         305        36          24                       83.6
Kindergarten       710       154          13                       81.0
Retail_Daily      5387      7003        2401                       36.4
Retail_Non-Daily  1970      2413         321                       41.9
Leisure           5942      3071         406                       63.1
  ..  the same six-category set on 68.1 % of the kept buildings

the most frequent disagreements (rules class -> LLM class), of 19,879:
services  ->  normal office                                                                               1925
services  ->  retail small scale                                                                          1518
services  ->  customer oriented services                                        

## Where this leaves us

- **Bosserhof, exact class: the LLM is ahead.** 64.8 % vs 59.6 % on 1,025 units: 159 units only the LLM gets right, 106 only the rules (McNemar p = 0.0014). On 845 of these units the truth is the old LLM answer, and the whole lead sits there (73.0 vs 65.8, p < 0.001). Where the validator rejected the old answer (180 units) the rules are ahead, 30.6 vs 26.1, but not significantly (p = 0.29). The LLM gives the rejected class again on 68 of them, the rules on 47. Neither split is neutral (see the top).
- **Bosserhof, main group: no significant difference.** 78.5 % vs 76.6 % (p = 0.19). The rules answer a bare main group more often (315 vs 203 units). They give no class on 10 scored units, which carry residential or supply-building ALKIS codes that the rules mark as no activity.
- **Activities: no significant difference.** Under the old merge (the headline) F1 is 67.1 vs 65.4: +1.6 points from the unrounded values, 95 % interval −0.4 .. +3.7. The sign favours the LLM in every subset. Under FINAL's map it is 71.2 vs 72.6 (−1.3; −3.7 .. +1.0). The totals hide offsetting categories:
  - The LLM is better on Retail_Non-Daily (F1 0.74 vs 0.63; the rules find only half).
  - The rules are better on Leisure.
  - Both over-predict Retail_Daily under the old merge (precision 0.24), mostly through errands → Retail_Daily.
- **Sites help the rules**, a little but significantly: +1.5 exact points, +1.4 F1.
- **The sample is POI-heavy.**
  - POI tags decide 76.8 % of the scored units but 42.7 % of the kept buildings. The ALKIS code decides 22.7 % against 54.5 %.
  - On the code-decided units both are weak and level: exact 41.2 vs 38.2, by headline 62.2 vs 63.5.
  - So this sample says little about the half of the region the rules decide by code.
- **Across the region** the two give the same class on 50 % of the kept buildings and the same main group on 67 %. The largest disagreements are the rules' generic `services` against the LLM's normal office, retail (small-scale) and customer-oriented services. Two of those three stay inside the Services main group.

A neutral verdict needs truth that neither classifier's answers seeded, such as a blind sample, and repeated LLM runs.
